# 🚀 Giai Đoạn 1: Domain Generalization (Pre-training) - Nhận diện Ổ Gà

Notebook này thiết lập pipeline huấn luyện YOLOv8s tối ưu hóa hiệu năng cao trên máy trạm cấu hình **RTX 3060 (12GB VRAM)** và **64GB System RAM**. 

### Mục tiêu chính:
1. **Tối đa hóa Throughput**: Tận dụng 64GB RAM để lưu trữ bộ nhớ cache ảnh (`cache='ram'`), triệt tiêu I/O Bottleneck.
2. **Khả năng chống lỗi (Auto-Resume)**: Tự động phát hiện checkpoint cuối cùng (`last.pt`) để khôi phục trạng thái huấn luyện nếu có sự cố mất điện hoặc crash kernel.
4. **Domain Generalization**: Sử dụng kỹ thuật Augmentation cao cấp (`mosaic=1.0`, `mixup=0.1`) nhằm nâng cao khả năng kháng nhiễu.

## 🩺 Cell 1: Setup Môi Trường


In [ ]:
# --- IMPORT CÁC THƯ VIỆN CẦN THIẾT --- 
import os
import sys
import torch
import ultralytics
from ultralytics import YOLO

# Vô hiệu hóa Weights & Biases (WandB) để tránh yêu cầu đăng nhập/API key phiền phức
os.environ["WANDB_DISABLED"] = "true"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"

print(f"🐍 Phiên bản Python: {sys.version}")
print(f"🔥 Phiên bản PyTorch: {torch.__version__}")
print(f"🛠️ Phiên bản Ultralytics: {ultralytics.__version__}")

# --- KIỂM TRA GPU WORKSTATION --- 
print("\n🖥️ Kiểm tra thông tin phần cứng GPU hiện tại:")
# Chạy lệnh nvidia-smi để kiểm tra driver, CUDA version và dung lượng VRAM thực tế của RTX 3060 (12GB)
!nvidia-smi

cuda_available = torch.cuda.is_available()
print(f"\n🤖 CUDA có khả dụng (GPU): {cuda_available}")
if cuda_available:
    print(f"✅ Đang sử dụng GPU: {torch.cuda.get_device_name(0)}")
    print(f"⚙️ Phiên bản CUDA của PyTorch: {torch.version.cuda}")
else:
    print("❌ CẢNH BÁO: PyTorch chưa nhận được CUDA GPU. Vui lòng cài đặt lại phiên bản PyTorch CUDA!")


## ⚡ Cell 2: Huấn Luyện Tối Đa Hiệu Năng (Max-Performance Training) & Auto-Resume

In [ ]:
import os
from ultralytics import YOLO

# ========================================== 
# 1. ĐỊNH VỊ FILE CẤU HÌNH DATASET
# ========================================== 
# Định nghĩa danh sách các đường dẫn dự phòng để đảm bảo tìm thấy dataset.yaml dù chạy từ bất kỳ thư mục nào
yaml_candidates = [
    "data/processed/combined_pothole/dataset.yaml",
    "../data/processed/combined_pothole/dataset.yaml"
]
dataset_yaml = None
for path in yaml_candidates:
    if os.path.exists(path):
        dataset_yaml = path
        break

if not dataset_yaml:
    raise FileNotFoundError("❌ Không tìm thấy file dataset.yaml. Hãy kiểm tra lại thư mục làm việc!")
else:
    print(f"✅ Đã tìm thấy file dataset.yaml tại: {os.path.abspath(dataset_yaml)}")
    # Tự động cập nhật path trong file YAML thành đường dẫn tuyệt đối trên máy hiện tại
    import yaml
    with open(dataset_yaml, 'r', encoding='utf-8') as f:
        yaml_content = yaml.safe_load(f)
    abs_dataset_dir = os.path.abspath(os.path.dirname(dataset_yaml))
    yaml_content['path'] = abs_dataset_dir.replace('\\', '/')
    with open(dataset_yaml, 'w', encoding='utf-8') as f:
        yaml.safe_dump(yaml_content, f, allow_unicode=True)
    print(f"🔄 Đã tự động cập nhật 'path' trong dataset.yaml thành: {yaml_content['path']}")

# ========================================== 
# 2. AUTO-RESUME LOGIC (KHẢ NĂNG CHỊU LỖI)
# ========================================== 
# Kiểm tra checkpoint lưu trữ lần chạy trước đó
# Khi ta đặt project='runs/detect' và name='base_model', checkpoint cuối cùng luôn ở weights/last.pt
checkpoint_candidates = [
    "runs/detect/base_model/weights/last.pt",
    "../runs/detect/base_model/weights/last.pt",
    "runs/detect/train/weights/last.pt",
    "../runs/detect/train/weights/last.pt"
]

last_checkpoint = None
for cp_path in checkpoint_candidates:
    if os.path.exists(cp_path):
        last_checkpoint = cp_path
        break

# ========================================== 
# 3. KHỞI CHẠY HUẤN LUYỆN (ÉP XUNG PHẦN CỨNG + TỐI ƯU HỘI TỤ)
# ========================================== 
if last_checkpoint:
    print(f"🔄 [RESUME] Phát hiện checkpoint cuối cùng tại: {last_checkpoint}")
    print("⏳ Tiến hành tiếp tục huấn luyện từ epoch bị ngắt quãng...")
    # Load trực tiếp checkpoint bị gián đoạn
    model = YOLO(last_checkpoint)
    # Khi resume=True, YOLOv8 sẽ tự động nạp lại toàn bộ siêu tham số cũ từ checkpoint
    results = model.train(resume=True)
else:
    print("🆕 [NEW RUN] Không tìm thấy checkpoint cũ. Khởi tạo mô hình mới yolov8s.pt...")
    # Khởi tạo mô hình yolov8s (size S cân bằng cực tốt giữa mAP và FPS thực tế)
    model = YOLO("yolov8s.pt")
    
    print("🔥 Đang chạy tiến trình huấn luyện tối đa hóa hiệu năng phần cứng...")
    results = model.train(
        # --- Đường dẫn và đầu ra ---
        data=dataset_yaml,
        project="runs/detect",
        name="base_model",
        exist_ok=True,             # Cho phép ghi đè vào thư mục này để phục vụ resume dễ dàng

        # --- Siêu tham số ép xung phần cứng (RTX 3060 12GB + 64GB RAM) ---
        cache=False,              # BẮT BUỘC: Đưa toàn bộ ~15k ảnh lên System RAM. Với 64GB RAM, việc này hoàn toàn khả thi.
                                  # Xóa bỏ nghẽn cổ chai I/O ổ cứng, chuyển dữ liệu trực tiếp từ RAM sang VRAM nhanh nhất.
        workers=0,                # Đặt workers=0 để tránh treo trên Windows để tải dữ liệu song song (Data Loading threads).
        batch=16,                 # Batch size = 32 tối ưu hóa dung lượng 12GB VRAM (giữ VRAM ở mức ~9-10GB, tránh OOM).
        imgsz=640,                # Kích thước ảnh đầu vào tiêu chuẩn 640x640.
        device=0,                 # Định hướng huấn luyện trên GPU đầu tiên.
        amp=False,                 # Automatic Mixed Precision (FP16) - Giúp giảm tải dung lượng VRAM và tăng tốc tính toán của Tensor Cores.

        # --- Tham số thuật toán & Hội tụ ---
        epochs=100,               # Huấn luyện trong 100 epochs cho giai đoạn Domain Generalization.
        patience=20,              # Tự động dừng sớm nếu mAP@50-95 không cải thiện sau 20 epochs.
        optimizer="auto",         # YOLO sẽ tự động chọn optimizer tốt nhất (AdamW là lựa chọn tối ưu cho sự hội tụ nhanh).

        # --- Augmentation nâng cao cho Domain Generalization (Kháng nhiễu đường sá) ---
        mosaic=1.0,               # Bật tối đa tính năng Mosaic (100% tỷ lệ ảnh đầu vào được ghép từ 4 ảnh khác nhau).
                                  # Giúp mô hình học được bối cảnh ổ gà đa dạng ở các kích thước và vị trí bất kỳ.
        mixup=0.1,                # Trộn hai hình ảnh ngẫu nhiên với tỷ lệ mờ nhất định (10% cơ hội).
                                  # Giúp mô hình có khả năng chống overfitting và domain generalization tốt hơn giữa các quốc gia.
        close_mosaic=15,          # Tắt tính năng Mosaic ở 15 epochs cuối cùng.
                                  # Lý do: Trong những epoch cuối, tắt Mosaic giúp mô hình học phân phối bounding box thực tế của ảnh gốc,
                                  # tinh chỉnh độ khít và phân bố chính xác của nhãn.
    )
    print("🎉 Quá trình huấn luyện đã kết thúc thành công!")


## 🎯 Cell 3: Đánh Giá Metrics Khách Quan (Validation)

In [ ]:
import os
from ultralytics import YOLO

# ========================================== 
# 1. ĐƯỜNG DẪN ĐẾN MODEL TỐT NHẤT (BEST WEIGHTS)
# ========================================== 
best_weight_candidates = [
    "runs/detect/base_model/weights/best.pt",
    "../runs/detect/base_model/weights/best.pt"
]
best_model_path = None
for path in best_weight_candidates:
    if os.path.exists(path):
        best_model_path = path
        break

if not best_model_path:
    raise FileNotFoundError("❌ Không tìm thấy file best.pt. Vui lòng kiểm tra lại quá trình huấn luyện!")

# Tìm lại file cấu hình dataset.yaml
yaml_candidates = [
    "data/processed/combined_pothole/dataset.yaml",
    "../data/processed/combined_pothole/dataset.yaml"
]
dataset_yaml = None
for path in yaml_candidates:
    if os.path.exists(path):
        dataset_yaml = path
        break

# ========================================== 
# 2. CHẠY TIẾN TRÌNH ĐÁNH GIÁ (VALIDATION)
# ========================================== 
print(f"🎯 Đang load mô hình tốt nhất từ: {best_model_path}")
model_eval = YOLO(best_model_path)

print("📊 Đang tiến hành đánh giá mô hình trên tập Validation...")
val_results = model_eval.val(
    data=dataset_yaml,
    device=0,
    batch=32,             # Tận dụng 12GB VRAM để chạy validation tốc độ cao
    imgsz=640
)

# Trích xuất các chỉ số quan trọng từ kết quả
metrics = val_results.results_dict
map50 = metrics.get('metrics/mAP50(B)', 0.0)
map50_95 = metrics.get('metrics/mAP50-95(B)', 0.0)

print("\n" + "="*60)
print("🏆 KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP VAL (VAL METRICS COCO):")
print(f"📈 mAP@50 (IoU=0.50)      : {map50:.4f} ({map50 * 100:.2f}%)")
print(f"📈 mAP@50-95 (IoU=0.50:0.95): {map50_95:.4f} ({map50_95 * 100:.2f}%)")
print("="*60)

# ========================================== 
# 3. HƯỚNG DẪN ĐỌC CHỈ SỐ CHO KỸ SƯ
# ========================================== 
print("""
💡 HƯỚNG DẪN ĐỌC CHỈ SỐ METRICS:
1. mAP@50 (Mean Average Precision tại ngưỡng IoU = 0.50):
   - Ý nghĩa: Đo khả năng phát hiện ổ gà khi hộp dự đoán và hộp nhãn thực tế khớp nhau từ 50% trở lên.
   - Đánh giá: Phản ánh khả năng nhận diện cơ bản (độ nhạy cao, hạn chế bỏ sót ổ gà). Con số này thường khá cao.

2. mAP@50-95 (mAP trung bình từ ngưỡng IoU = 0.50 đến 0.95 với bước nhảy 0.05):
   - Ý nghĩa: Đo lường độ khít, độ chính xác vị trí của hộp dự đoán ở nhiều mức độ khắt khe khác nhau.
   - Đánh giá: Đây là thước đo tiêu chuẩn MLOps cao nhất. Điểm mAP@50-95 cao chứng tỏ mô hình không chỉ nhận diện
     được ổ gà mà còn khoanh vùng rất chính xác và sát thực tế, cực kỳ quan trọng cho các ứng dụng thực tế trên xe tự hành.
""")


## 📦 Cell 4: Xuất Bản Mô Hình (Export to ONNX)

In [ ]:
import os
from ultralytics import YOLO

# ========================================== 
# 1. ĐỊNH VỊ MODEL BEST WEIGHTS
# ========================================== 
best_weight_candidates = [
    "runs/detect/base_model/weights/best.pt",
    "../runs/detect/base_model/weights/best.pt"
]
best_model_path = None
for path in best_weight_candidates:
    if os.path.exists(path):
        best_model_path = path
        break

if not best_model_path:
    raise FileNotFoundError("❌ Không tìm thấy file best.pt để tiến hành xuất bản!")

print(f"🎯 Đang nạp mô hình tốt nhất từ: {best_model_path}")
model_export = YOLO(best_model_path)

# ========================================== 
# 2. XUẤT MODEL SANG ĐỊNH DẠNG ONNX
# ========================================== 
print("\n📦 Bắt đầu chuyển đổi mô hình sang định dạng ONNX...")
onnx_path = model_export.export(
    format="onnx",        # Định dạng đích chuyển đổi
    dynamic=True,         # Bật Dynamic Shapes (cho phép batch_size và image_size thay đổi linh hoạt lúc deploy)
    simplify=True         # Tối ưu hóa đồ thị ONNX (loại bỏ các node thừa, gộp các phép tính toán)
)

print("\n" + "="*60)
print("🎉 ĐÃ XUẤT BẢN MÔ HÌNH THÀNH CÔNG!")
print(f"💾 Đường dẫn lưu file ONNX: {onnx_path}")
print("""
💡 LỢI ÍCH CỦA FILE ONNX:
1. Độc lập nền tảng: Có thể chạy suy luận bằng ONNX Runtime trên CPU/GPU của bất kỳ ngôn ngữ nào (C++, Rust, C#).
2. Tương thích Edge Devices: Chuẩn bị sẵn sàng cho việc convert sang TensorRT (NVIDIA Jetson) hoặc OpenVINO (Intel).
3. Deploy Web/Mobile: Cho phép chạy suy luận trực tiếp trên trình duyệt hoặc các ứng dụng nhúng.
""")
print("="*60)
